# Finding Trailers of the Movies

In this notebook, we identify and collect official movie trailers from the KinoCheck.com.
The purpose of this step is to build a reliable list of trailers that can later be matched to movies and enriched with additional data.

#### Why KinoCheck is used

The main challenge at this point is simply finding the correct trailer for each movie and doing it most cost-efficient way.

KinoCheck provides a curated source of official trailers, which makes it easier to:
- avoid fan-made or unrelated videos
- work with consistent and recognizable trailer titles
- reduce the number of API requests

#### Scope of the data

Only trailers that meet the following criteria are collected:
- released between 2006 and 2016
- published on the KinoCheck.com channel
- clearly identifiable as movie trailers

For each trailer, only minimal metadata is stored:
- year
- original title
- cleaned title for later matching

In [2]:
# Install the needed Python packages
%pip -q install happybase thriftpy2 google-api-python-client vaderSentiment tqdm pandas

Note: you may need to restart the kernel to use updated packages.


In [3]:
# Imports + config
import os
import re
import time
import hashlib
from datetime import datetime

import happybase
import pandas as pd
from tqdm import tqdm
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

In [4]:
YOUTUBE_API_KEY = os.getenv("YOUTUBE_API_KEY")
print("Key loaded:", bool(YOUTUBE_API_KEY))
if not YOUTUBE_API_KEY:
    raise ValueError("Missing YOUTUBE_API_KEY. Set it as an env var or paste it in directly.")

Key loaded: True


We also establish a connection to HBase, since the collected trailer data is stored directly in a database table.


In [5]:
# close the old connection
try:
    connection.close()
except Exception:
    pass

# new connection
connection = happybase.Connection(host="hbase", port=9090)
connection.open()

print(connection.tables())

[b'movie_success_rate_raw']


### Configuration and table setup

At the beginning, the names of the relevant HBase tables and columns are defined.

- `movie_success_rate_raw` contains the original movie success data
- `kinocheck_videos` will store the trailers collected in this notebook

In [6]:
# === CONFIGURATION ===

MOVIES_TABLE = "movie_success_rate_raw"
KINOCHECK_VIDEOS_TABLE = "kinocheck_videos"

TITLE_COL = b"cf:title"
YEAR_COL  = b"cf:year"

print("Using movies table:", MOVIES_TABLE)
print("Title column:", TITLE_COL)
print("Year column:", YEAR_COL)


Using movies table: movie_success_rate_raw
Title column: b'cf:title'
Year column: b'cf:year'


### Creating the KinoCheck videos table

This code checks whether the HBase table `kinocheck_videos` already exists.
If it does not exist, the table is created with a column family for YouTube-related data.

If the table already exists, it is reused.
After that, a table reference is created so trailer data can be written in the next steps.


In [7]:
existing = [t.decode() for t in connection.tables()]

# HBase Tabelle für KinoCheck-Videos anlegen

existing_tables = [t.decode() for t in connection.tables()]
if KINOCHECK_VIDEOS_TABLE not in existing_tables:
    connection.create_table(
        KINOCHECK_VIDEOS_TABLE,
        {
            "yt": dict(),   # YouTube-spezifische Infos
        }
    )
    print("Created table:", KINOCHECK_VIDEOS_TABLE)
else:
    print("Table already exists:", KINOCHECK_VIDEOS_TABLE)

kinocheck_videos = connection.table(KINOCHECK_VIDEOS_TABLE)

Created table: kinocheck_videos


### Connecting to the YouTube Data API

This initializes the YouTube Data API client using the provided API key.

In [8]:
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from datetime import datetime

youtube = build(
    "youtube",
    "v3",
    developerKey=YOUTUBE_API_KEY
)

### Finding and verifying the KinoCheck channel

The YouTube API is used to search for the channel "KinoCheck.com" and retrieve its channel ID.
The result is then compared with a known, previously verified channel ID.

This check ensures that the data is collected from the correct YouTube channel.


In [9]:
# KinoCheck Channel ID über die YouTube API holen

KINOCHECK_CHANNEL_NAME = "KinoCheck.com"

channel_search_response = youtube.search().list(
    part="snippet",
    q=KINOCHECK_CHANNEL_NAME,
    type="channel",
    maxResults=1
).execute()

KINOCHECK_CHANNEL_ID = channel_search_response["items"][0]["id"]["channelId"]
print("Gefundene KinoCheck Channel ID:", KINOCHECK_CHANNEL_ID)

# Erwartete ID (Ich habe die ID schon über Postman geholt und gleiche nur noch ab)
KINOCHECK_CHANNEL_ID_KNOWN = "UCLRlryMfL8ffxzrtqv0_k_w"
print("Erwartete Channel ID:", KINOCHECK_CHANNEL_ID_KNOWN)

assert KINOCHECK_CHANNEL_ID == KINOCHECK_CHANNEL_ID_KNOWN, "Channel ID stimmt nicht mit der erwarteten überein!"


Gefundene KinoCheck Channel ID: UCLRlryMfL8ffxzrtqv0_k_w
Erwartete Channel ID: UCLRlryMfL8ffxzrtqv0_k_w


### Getting the uploads playlist ID

Every YouTube channel has an uploads playlist that contains all videos published on that channel.
Here, the playlist ID for the KinoCheck channel is retrieved via the YouTube API.

This playlist is later used to fetch the channel’s videos efficiently.
The ID is also compared with a known value to make sure the correct playlist is used.


In [10]:
# KinoCheck Uploads Playlist ID über die YouTube API holen
# Über die Upload ID bekommt man mit einem Aufruf 50 Video IDs mit einem Aufruf

channels_response = youtube.channels().list(
    part="contentDetails",
    id=KINOCHECK_CHANNEL_ID
).execute()

UPLOADS_PLAYLIST_ID = channels_response["items"][0]["contentDetails"]["relatedPlaylists"]["uploads"]
print("Gefundene Uploads Playlist ID:", UPLOADS_PLAYLIST_ID)

# Erwartete Uploads Playlist ID (auch bereits von Postman bekannt)
UPLOADS_PLAYLIST_ID_KNOWN = "UULRlryMfL8ffxzrtqv0_k_w"
print("Erwartete Uploads Playlist ID:", UPLOADS_PLAYLIST_ID_KNOWN)

assert UPLOADS_PLAYLIST_ID == UPLOADS_PLAYLIST_ID_KNOWN, "Uploads Playlist ID stimmt nicht mit der erwarteten überein!"


Gefundene Uploads Playlist ID: UULRlryMfL8ffxzrtqv0_k_w
Erwartete Uploads Playlist ID: UULRlryMfL8ffxzrtqv0_k_w


### Normalizing and filtering trailer titles

Trailer titles on YouTube often include additional words such as
“Official”, “Movie”, or “Trailer”.

These words are removed and the titles are:
- converted to lowercase
- normalized to a consistent format

The cleaned title is stored in the column `title_clean` and is later used to match trailers with movies from the movie success dataset.
The function returns the cleaned title, the extracted year, and a flag indicating whether the video should be skipped.

In [11]:
import re

# Dinge, die KEINE normalen Trailer sind (rausfiltern)
SKIP_PHRASES = ["the best","all new","all trailers","trailer compilation","clips","clip",
                "scenes","scene","featurette","spot","preview","behind the scenes","interview","recap","explained","news",
]

SERIES_KEYWORDS = [
    "season",
    "episode",
    "series",
]

def normalize_title_english(title: str):
    """
    Normalisiert englische KinoCheck.com Filmtitel.

    Regeln:
    - Skip: Compilations/Clips/Scenes/Featurettes/Spots/Series
    - Jahr extrahieren: (2026) -> year=2026
    - Titel schneiden:
        1) alles ab 'official' (inklusive 'official') entfernen
        2) falls kein 'official' vorkommt: alles ab 'trailer' entfernen
    - Whitespace/Sonderzeichen bereinigen

    Rückgabe: (title_clean, year, skip_flag)
    """
    if not title:
        return None, None, True

    t = title.lower()

    # Skip-Regeln
    for phrase in SKIP_PHRASES:
        if phrase in t:
            return None, None, True

    for kw in SERIES_KEYWORDS:
        if kw in t:
            return None, None, True

    # Jahr extrahieren
    year_match = re.search(r"\(\s*(\d{4})\s*\)", t)
    year = int(year_match.group(1)) if year_match else None

    # 1) Erst nach "official" schneiden
    if re.search(r"\bofficial\b", t):
        t = re.split(r"\bofficial\b", t, maxsplit=1)[0]
    # 2) Wenn kein official drin: nach "trailer" schneiden
    elif re.search(r"\btrailer\b", t):
        t = re.split(r"\btrailer\b", t, maxsplit=1)[0]

    # Jahresangaben entfernen (nach Extraktion)
    t = re.sub(r"\(\s*\d{4}\s*\)", "", t)

    # Sonderzeichen & Whitespace bereinigen
    t = re.sub(r"[|–—]", "-", t)
    t = re.sub(r"\s+", " ", t).strip(" -:")

    if not t:
        return None, None, True

    return t, year, False


### Collecting and saving the trailers

Here, we go through all videos from the KinoCheck uploads playlist step by step.
For each video, the title is cleaned and checked to make sure it really represents a movie trailer.

Only trailers that:
- have a valid year in the title
- fall between 2006 and 2016
- are not clips, scenes, or compilations

are saved.

The matching trailers are written to the `kinocheck_videos` table in batches to keep the process efficient.
The loop stops once enough trailers are stored or there are no more videos left in the playlist.

In [12]:
import time
import happybase

HBASE_HOST = "hbase"
HBASE_PORT = 9090

SOURCE_PLAYLIST_ID = UPLOADS_PLAYLIST_ID  # deine Uploads Playlist
TARGET_TABLE = "kinocheck_videos"

MIN_YEAR = 2006
MAX_YEAR = 2016
MAX_TO_SAVE = 1500  # ruhig höher, um genug Treffer zu bekommen

def ensure_table(conn, name):
    if name.encode() not in conn.tables():
        conn.create_table(name, {"yt": dict()})
    return conn.table(name)

def store_only_year_range_trailers(youtube, uploads_playlist_id, min_year=2006, max_year=2016, max_to_save=1500):
    """
    Speichert NUR Trailer, deren extrahiertes Jahr im Bereich [min_year, max_year] liegt,
    und nur wenn skip=false. Schreibt in neue Tabelle kinocheck_videos_2006_2016.
    """
    conn = happybase.Connection(host=HBASE_HOST, port=HBASE_PORT, timeout=20000)
    conn.open()
    table = ensure_table(conn, TARGET_TABLE)

    saved = 0
    next_page_token = None

    try:
        while saved < max_to_save:
            resp = youtube.playlistItems().list(
                part="snippet,contentDetails",
                playlistId=uploads_playlist_id,
                maxResults=50,
                pageToken=next_page_token
            ).execute()

            items = resp.get("items", [])
            if not items:
                print("Ende der Playlist erreicht.")
                break

            with table.batch(batch_size=200) as b:
                for item in items:
                    video_id = item["contentDetails"]["videoId"]
                    title = item["snippet"]["title"]

                    title_clean, year, skip = normalize_title_english(title)

                    # Jahr muss vorhanden und im Bereich sein
                    if year is None or not (min_year <= year <= max_year):
                        continue

                    # Nur echte Trailer (skip=false)
                    if skip:
                        continue

                    # Schreiben
                    b.put(video_id.encode("utf-8"), {
                        b"yt:video_id": video_id.encode("utf-8"),
                        b"yt:title": title.encode("utf-8", errors="ignore"),
                        b"yt:title_clean": title_clean.encode("utf-8", errors="ignore"),
                        b"yt:year": str(year).encode("utf-8"),
                        b"yt:skip": b"false",
                    })
                    saved += 1
                    if saved >= max_to_save:
                        break

            next_page_token = resp.get("nextPageToken")
            if not next_page_token:
                break

            time.sleep(0.1)

        print(f"Gespeichert in {TARGET_TABLE}: {saved}")
        return saved
    finally:
        conn.close()

# --- Run ---
store_only_year_range_trailers(
    youtube=youtube,
    uploads_playlist_id=SOURCE_PLAYLIST_ID,
    min_year=MIN_YEAR,
    max_year=MAX_YEAR,
    max_to_save=MAX_TO_SAVE
)


Gespeichert in kinocheck_videos: 499


499

### Verification of the stored trailer data

We read a small sample of rows from the `kinocheck_videos` table and print them.
It is used to confirm that trailers were stored correctly and that the extracted year and cleaned title look as expected.


In [13]:
import happybase

conn = happybase.Connection(host="hbase", port=9090, timeout=20000)
conn.open()
t = conn.table("kinocheck_videos")

for rk, data in t.scan(limit=20, columns=[b"yt:year", b"yt:title", b"yt:title_clean"]):
    year = data.get(b"yt:year", b"").decode()
    title = data.get(b"yt:title", b"").decode("utf-8", errors="ignore")
    clean = data.get(b"yt:title_clean", b"").decode("utf-8", errors="ignore")
    print(year, rk.decode(), "->", title, "|", clean)

conn.close()


2016 -1U8gmQ4PL4 -> JACK REACHER 2 Trailer 2 (2016) | jack reacher 2
2016 -GQjm4TrQIM -> Alice Through the Looking Glass Trailer 3 (2016) | alice through the looking glass
2016 -VsEFme-HLE -> RIDE ALONG 2 Official Trailer (2016) | ride along 2
2015 -gD-ywBXQns -> THE LAST WITCH HUNTER Official Trailer (2015) | the last witch hunter
2016 -sDW0173y1Q -> THE PURGE 3 Election Year Trailer 2 (2016) | the purge 3 election year
2016 0A5rvlALfBk -> THE HUNTSMAN: Winter's War Official Trailer 2 (2016) | the huntsman: winter's war
2016 0fITmuqh4g8 -> ASSASSIN'S CREED Movie Trailer 3 (2016) | assassin's creed movie
2016 0la2m76py10 -> Teenage Mutant Ninja Turtles 2 ALL Trailer (2016) | teenage mutant ninja turtles 2 all
2016 0neX1W1p7BU -> MASCOTS Trailer (2016) | mascots
2016 0pbsqakNL1c -> MOMENTUM Official Trailer (2016) | momentum
2015 16jSEcCTO8o -> BLACK MASS Official Trailer (2015) | black mass
2016 17Mj1VQBvss -> FINDING DORY Trailer 3 (2016) | finding dory
2008 1EEhY44xF2I -> All MARVEL 